## Download Dataset

- https://arxiv.org/html/2509.09495v1
- https://huggingface.co/datasets/ComplexDataLab/OpenFake
- https://huggingface.co/docs/datasets/en/loading

In [ ]:
from datasets import load_dataset,Image
import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset('ComplexDataLab/OpenFake', split='train', streaming=True)
dataset = dataset.cast_column("image", Image(decode=False)) # This stops the auto-crash

In [ ]:
sample = next(iter(dataset))

for k, v in sample.items():
    print(f"{k}: {v}")
    print("-" * 40)

In [ ]:
import matplotlib.pyplot as plt
import io
from PIL import Image as PILImage

samples = []

# collect first 25 samples
for i, sample in enumerate(dataset):
    samples.append(sample)
    if i == 24:
        break

# plot grid
fig, axes = plt.subplots(5, 5, figsize=(12, 12))

for i, ax in enumerate(axes.flatten()):
    sample = samples[i]

    try:
        # decode manually (IMPORTANT)
        img_bytes = sample["image"]["bytes"]
        img = PILImage.open(io.BytesIO(img_bytes))

        if img.mode != "RGB":
            img = img.convert("RGB")

        ax.imshow(img)

    except Exception:
        # fallback if image is broken
        ax.text(0.5, 0.5, "Error", ha='center', va='center')
        ax.axis("off")
        continue

    # prompt + label + model
    prompt = sample.get("prompt", "")
    prompt = prompt[:40] + "..." if len(prompt) > 40 else prompt

    ax.set_title(
        f"{sample.get('label','')} | {sample.get('model','')}\n{prompt}",
        fontsize=7
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

## Simple EDA

In [ ]:
from collections import Counter

label_counts = Counter()
model_counts = Counter()

for i, sample in enumerate(dataset):
    label_counts[sample["label"]] += 1
    model_counts[sample["model"]] += 1

    if i == 100:
        break

print("Labels:", label_counts)
print("Models:", model_counts)

## Download Subset

In [ ]:
!rm -r /content/openfake_subset

In [ ]:
import os
import csv
import io
from tqdm.auto import tqdm
from datasets import load_dataset, Image
from PIL import Image as PILImage

# =========================
# 1. PATHS
# =========================
base_path = "/content/openfake_subset"
image_dir = f"{base_path}/images"

os.makedirs(image_dir, exist_ok=True)

csv_path = f"{base_path}/metadata.csv"

# =========================
# 2. TARGETS
# =========================
targets = {
    "real": 3000,
    "midjourney": 3000,
    "stable_diffusion": 3000
}

counts = {
    "real": 0,
    "midjourney": 0,
    "stable_diffusion": 0
}

# =========================
# 3. LOAD DATASET (STREAMING + SAFE MODE)
# =========================
dataset = load_dataset(
    "ComplexDataLab/OpenFake",
    split="train",
    streaming=True
)

# VERY IMPORTANT → prevent auto decode crash
dataset = dataset.cast_column("image", Image(decode=False))

# =========================
# 4. CSV SETUP
# =========================
csv_file = open(csv_path, "w", newline="", encoding="utf-8")
writer = csv.writer(csv_file)
writer.writerow(["image_path", "prompt", "label", "model"])

# =========================
# 5. PROCESS LOOP
# =========================
pbar = tqdm(total=9000)

img_id = 0
skip_count = 0

iterator = iter(dataset)

while True:
    try:
        example = next(iterator)

        model_name = example.get("model", "").lower()
        label_name = example.get("label", "").lower()
        prompt = example.get("prompt", "")

        category = None

        # =========================
        # CLASSIFY CATEGORY
        # =========================
        if label_name == "real":
            category = "real"

        elif "midjourney" in model_name:
            category = "midjourney"

        elif any(x in model_name for x in ["sd", "stable", "sdxl"]):
            category = "stable_diffusion"

        # =========================
        # SAVE IMAGE + CSV
        # =========================
        if category and counts[category] < targets[category]:
            try:
                img_data = example["image"]

                # 🔥 manual decoding (KEY FIX)
                img = PILImage.open(io.BytesIO(img_data["bytes"]))

                # fix RGBA / P / etc.
                if img.mode != "RGB":
                    img = img.convert("RGB")

                img_path = f"{image_dir}/{img_id}.jpg"
                img.save(img_path, "JPEG")

                # write metadata
                writer.writerow([img_path, prompt, category, model_name])

                counts[category] += 1
                img_id += 1
                pbar.update(1)


                if img_id % 1000 == 0 and img_id != 0:
                  print(f"\n💾 Saving checkpoint at {img_id} images...")

                  zip_path = f"/content/openfake_subset_{img_id}.zip"

                  import shutil
                  shutil.make_archive(
                      f"/content/openfake_subset_{img_id}",
                      'zip',
                      base_path
                  )

                  print(f"✅ Saved: {zip_path}")

            except Exception:
                skip_count += 1
                continue

        # =========================
        # STOP CONDITION
        # =========================
        if all(counts[k] >= targets[k] for k in targets):
            print("\n✅ DONE!")
            break

    except StopIteration:
        print("\n⚠️ Dataset ended before reaching target")
        break

    except Exception:
        skip_count += 1
        continue

# =========================
# 6. CLOSE FILES
# =========================
csv_file.close()
pbar.close()

# =========================
# 7. FINAL REPORT
# =========================
print("\n📊 Final counts:")
for k, v in counts.items():
    print(f"{k}: {v}")

print(f"\n⚠️ Skipped images: {skip_count}")
print(f"\n📁 CSV saved at: {csv_path}")
print(f"🖼️ Images saved in: {image_dir}")

## Download

In [ ]:
import shutil
shutil.make_archive("/content/openfake_subset", 'zip', "/content/openfake_subset")

In [ ]:
from google.colab import files
files.download("/content/openfake_subset")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/openfake_subset /content/drive/MyDrive/